In [2]:
import matplotlib.pyplot as plt
import numpy as np


# Numpy

## Numpy arrays

Representación de vectores y matrices (y tensores) en Python. Numpy es una librería que permite trabajar con arreglos multidimensionales de manera eficiente (pues delega el trabajo a código optimizado en C).

In [3]:
# Vectores

vector1 = np.array([1, 2, 3])
vector2 = np.array([4, 5, 6])

print("Suma (vector1 + vector2):", vector1 + vector2)
print("Resta (vector1 - vector2):", vector1 - vector2)
print("Producto punto (vector1 . vector2):", np.dot(vector1, vector2))
print("Norma de vector1 (L2):", np.linalg.norm(vector1))
print("Norma de vector1 (L1):", np.linalg.norm(vector1, ord=1))

Suma (vector1 + vector2): [5 7 9]
Resta (vector1 - vector2): [-3 -3 -3]
Producto punto (vector1 . vector2): 32
Norma de vector1 (L2): 3.7416573867739413
Norma de vector1 (L1): 6.0


In [4]:
# Matrices

mat1 = np.array([[1, 2], 
                 [3, 4]])
mat2 = np.array([[5, 6], 
                 [7, 8]])

print("Suma (mat1 + mat2):\n", mat1 + mat2)
print("Resta (mat1 - mat2):\n", mat1 - mat2)
print("Producto (elemento a elemento) (mat1 * mat2):\n", mat1 * mat2)  
print("Producto (mat1 @ mat2):\n", mat1 @ mat2)  # Multiplicación de matrices

Suma (mat1 + mat2):
 [[ 6  8]
 [10 12]]
Resta (mat1 - mat2):
 [[-4 -4]
 [-4 -4]]
Producto (elemento a elemento) (mat1 * mat2):
 [[ 5 12]
 [21 32]]
Producto (mat1 @ mat2):
 [[19 22]
 [43 50]]


In [5]:
# TRANFORMATIONS

def rotate(angle):
    """Returns the rotation matrix for a given angle in degrees."""
    rad = np.radians(angle)
    return np.array([[np.cos(rad), -np.sin(rad)],
                     [np.sin(rad),  np.cos(rad)]])
    
def sheer(shear_x, shear_y):
    """Returns the shear matrix for given shear factors in x and y directions."""
    return np.array([[1, shear_x],
                     [shear_y, 1]])
    
def scale(scale_x, scale_y):
    """Returns the scaling matrix for given scale factors in x and y directions."""
    return np.array([[scale_x, 0],
                     [0, scale_y]])

## Transformaciones lineales

Una transformación lineal lleva un vector $\mathbf{x}$ a otro mediante una matriz: $T(\mathbf{x}) = A\mathbf{x}$. 

Las columnas de $A$ indican adónde van los vectores de la base. Por eso una matriz $2\times2$ determina por completo cómo se transforma todo el plano.

In [6]:
import cv2
import ipywidgets as widgets
from ipywidgets import interact
from PIL import Image


### Rotación

La matriz $R(\theta)$ gira cada vector un ángulo $\theta$ sin cambiar su longitud ni los ángulos entre vectores. Una rotación lineal ocurre alrededor del origen. Para girar la imagen alrededor de su centro, el código añade una traslación: mueve conceptualmente el centro al origen, rota y lo devuelve. La operación completa es **afín**, no estrictamente lineal, porque incluye esa traslación.

In [7]:
with open("assets/dog.jpg", "rb") as f:
    img_pil = Image.open(f)
    img_array = np.array(img_pil)

def show_rotated_image(angle=0):
    R = rotate(angle)
    
    h, w = img_array.shape[:2]
    center = np.array([w / 2, h / 2])
    
    translation = center - np.dot(R, center)
    
    M = np.c_[R, translation]
    
    rotated_img = cv2.warpAffine(img_array, M, (w, h))
    
    plt.figure(figsize=(4, 4))
    plt.imshow(rotated_img)
    plt.title(f"Rotated by {angle}°")
    plt.axis('off')
    plt.show()

interact(
    show_rotated_image, 
    angle=widgets.IntSlider(min=-180, max=180, step=5, value=0, description='Angle:')
);

interactive(children=(IntSlider(value=0, description='Angle:', max=180, min=-180, step=5), Output()), _dom_cla…

### Cizallamiento

Un cizallamiento (*shear*) desplaza una coordenada en proporción a la otra. Por ejemplo, con $\begin{bmatrix}1 & k \\ 0 & 1\end{bmatrix}$ se obtiene $x'=x+ky$ y $y'=y$. Las líneas paralelas continúan siendo paralelas, cambian los ángulos y las longitudes.

In [8]:
# Sheer

def show_sheared_image(shear_x=0, shear_y=0):
    S = sheer(shear_x, shear_y)
    
    h, w = img_array.shape[:2]
    center = np.array([w / 2, h / 2])
    
    translation = center - np.dot(S, center)
    
    M = np.c_[S, translation]
    
    sheared_img = cv2.warpAffine(img_array, M, (w, h))
    
    plt.figure(figsize=(4, 4))
    plt.imshow(sheared_img)
    plt.title(f"Sheared by ({shear_x}, {shear_y})")
    plt.text(0, 700, f"Transformation Matrix:\n{S}", fontsize=8, color='blue', ha='left', va='top')
    plt.axis('off')
    plt.show()
    
interact(
    show_sheared_image, 
    shear_x=widgets.FloatSlider(min=-1, max=1, step=0.1, value=0, description='Shear X:'),
    shear_y=widgets.FloatSlider(min=-1, max=1, step=0.1, value=0, description='Shear Y:')
);

interactive(children=(FloatSlider(value=0.0, description='Shear X:', max=1.0, min=-1.0), FloatSlider(value=0.0…

### Escalamiento

Una matriz diagonal $S=\operatorname{diag}(s_x,s_y)$ multiplica por separado las componentes horizontal y vertical. Si $s_x=s_y$, el escalamiento es uniforme y conserva las proporciones; si son distintos, la figura se estira más en una dirección que en la otra. Igual que en la rotación de la imagen, se añade una traslación para usar su centro como punto fijo.

In [9]:
# Scale

def show_scaled_image(scale_x=1, scale_y=1):
    S = scale(scale_x, scale_y)
    
    h, w = img_array.shape[:2]
    center = np.array([w / 2, h / 2])
    
    translation = center - np.dot(S, center)
    
    M = np.c_[S, translation]
    
    scaled_img = cv2.warpAffine(img_array, M, (w, h))
    
    plt.figure(figsize=(4, 4))
    plt.imshow(scaled_img)
    plt.title(f"Scaled by ({scale_x}, {scale_y})")
    plt.axis('off')
    plt.text(600, 50, f"Transformation Matrix:\n{S}", fontsize=20, color='blue', ha='left', va='top')
    plt.show()
    
interact(
    show_scaled_image, 
    scale_x=widgets.FloatSlider(min=0.1, max=3, step=0.1, value=1, description='Scale X:'),
    scale_y=widgets.FloatSlider(min=0.1, max=3, step=0.1, value=1, description='Scale Y:')
);

interactive(children=(FloatSlider(value=1.0, description='Scale X:', max=3.0, min=0.1), FloatSlider(value=1.0,…

In [16]:
def get_transformed_plane_fn_with_shape(
    shapes: list[np.ndarray], show_shape_reference: bool = True
):

    def plot_transformed_plane(
        angle=0, scale_x=1.0, scale_y=1.0, shear_x=0.0, shear_y=0.0
    ):
        R = rotate(angle)
        S = scale(scale_x, scale_y)
        Sh = sheer(shear_x, shear_y)  # Example shear for demonstration

        # Combined transformation matrix: scale first, then rotate
        M = R @ S @ Sh

        # Because shape points are row vectors (N x 2), multiply by M.T
        # This is mathematically equivalent to: (M @ shape.T).T
        transformed_shapes = [shape @ M.T for shape in shapes]

        # Plot setup
        _, ax = plt.subplots(figsize=(6, 6))

        # Draw Cartesian axes through the origin
        ax.axhline(0, color="black", linewidth=1.2)
        ax.axvline(0, color="black", linewidth=1.2)
        ax.grid(True, linestyle=":", alpha=0.6)

        grid_bound = 8
        grid_values = np.arange(-grid_bound, grid_bound + 1, 1)

        for val in grid_values:
            h_line = np.array([[-grid_bound, val], [grid_bound, val]])
            v_line = np.array([[val, -grid_bound], [val, grid_bound]])

            h_trans = h_line @ M.T
            v_trans = v_line @ M.T

            # Style: make transformed axes thicker, regular grid lines thinner
            if val == 0:
                # Transformed X-axis (red) and Y-axis (green)
                ax.plot(
                    h_trans[:, 0],
                    h_trans[:, 1],
                    color="crimson",
                    linewidth=1.8,
                    label="Transformed X-axis" if val == 0 else "",
                )
                ax.plot(
                    v_trans[:, 0],
                    v_trans[:, 1],
                    color="forestgreen",
                    linewidth=1.8,
                    label="Transformed Y-axis" if val == 0 else "",
                )
            else:
                # Regular deformed grid lines
                ax.plot(
                    h_trans[:, 0],
                    h_trans[:, 1],
                    color="dodgerblue",
                    alpha=0.35,
                    linewidth=0.8,
                )
                ax.plot(
                    v_trans[:, 0],
                    v_trans[:, 1],
                    color="dodgerblue",
                    alpha=0.35,
                    linewidth=0.8,
                )

        ax.axhline(0, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
        ax.axvline(0, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)

        # Plot the original shape as a dashed reference outline
        if show_shape_reference:
          for shape in shapes:
              ax.plot(shape[:, 0], shape[:, 1], "k--", alpha=0.3, label="Original")

        # Plot the transformed shape
        for transformed_shape in transformed_shapes:
            ax.plot(
                transformed_shape[:, 0],
                transformed_shape[:, 1],
                "b-",
                linewidth=2,
                label="Transformed",
            )
            ax.fill(transformed_shape[:, 0], transformed_shape[:, 1], "skyblue", alpha=0.3)

        ax.set_aspect("equal")
        ax.set_xlim(-6, 6)
        ax.set_ylim(-6, 6)
        ax.set_xticks(np.arange(-6, 7, 1))
        ax.set_yticks(np.arange(-6, 7, 1))
        ax.set_title(
            f"Rotation: {angle}° | Scale: ({scale_x:.1f}, {scale_y:.1f}) | Shear: ({shear_x:.1f}, {shear_y:.1f})"
        )
        plt.text(10, 0, f"Transformation Matrix:\n{S}", fontsize=20, color='blue', ha='left', va='top')
        

        plt.show()
    
    return plot_transformed_plane

### Composición de transformaciones

Varias transformaciones lineales se combinan multiplicando sus matrices. En $M=R\,S\,Sh$, los vectores columna reciben primero $Sh$, después $S$ y finalmente $R$: el orden se lee de derecha a izquierda.

Los puntos de las figuras están almacenados aquí como filas. Por eso el código usa `shape @ M.T`, que equivale a aplicar $M$ a cada punto escrito como vector columna. La cuadrícula permite observar cómo una sola matriz transforma simultáneamente todos los puntos y direcciones del plano.

In [ ]:
house = np.array([
    [-1, -1],
    [ 1, -1],
    [ 1,  1],
    [ 0,  2],   # Roof peak
    [-1,  1],
    [-1, -1]    # Back to start to close the polygon
])

interact(
    get_transformed_plane_fn_with_shape([house]),
    angle=widgets.IntSlider(min=-180, max=180, step=5, value=0, description='Angle:'),
    scale_x=widgets.FloatSlider(min=0.1, max=3.0, step=0.1, value=1.0, description='Scale X:'),
    scale_y=widgets.FloatSlider(min=0.1, max=3.0, step=0.1, value=1.0, description='Scale Y:'),
    shear_x=widgets.FloatSlider(min=-1.0, max=1.0, step=0.1, value=0.0, description='Shear X:'),
    shear_y=widgets.FloatSlider(min=-1.0, max=1.0, step=0.1, value=0.0, description='Shear Y:')
);

interactive(children=(IntSlider(value=0, description='Angle:', max=180, min=-180, step=5), FloatSlider(value=1…

## Autovalores y autovectores

Un vector no nulo $\mathbf{v}$ es autovector de una matriz $A$ si $A\mathbf{v}=\lambda\mathbf{v}$. La transformación no cambia su dirección (aunque puede invertir su sentido): solamente lo escala por el autovalor $\lambda$.

Para $S=\operatorname{diag}(s_x,s_y)$, los ejes $\mathbf{e}_1=(1,0)$ y $\mathbf{e}_2=(0,1)$ son autovectores con autovalores $s_x$ y $s_y$. El vector $(1,1)$ del siguiente ejemplo solo es autovector cuando $s_x=s_y$ (junto con $(1,-1)$, $(-1,1)$ y $(-1,-1)$).

In [19]:
scale_eigenvectors = np.array([[0, 0], [1,1]]) # eigenvectors for scaling along the principal axes

interact(
    get_transformed_plane_fn_with_shape([scale_eigenvectors]),
    angle=widgets.IntSlider(min=-180, max=180, step=5, value=0, description='Angle:'),
    scale_x=widgets.FloatSlider(min=0.1, max=3.0, step=0.1, value=1.0, description='Scale X:'),
    scale_y=widgets.FloatSlider(min=0.1, max=3.0, step=0.1, value=1.0, description='Scale Y:'),
    shear_x=widgets.FloatSlider(min=-1.0, max=1.0, step=0.1, value=0.0, description='Shear X:'),
    shear_y=widgets.FloatSlider(min=-1.0, max=1.0, step=0.1, value=0.0, description='Shear Y:')
);

interactive(children=(IntSlider(value=0, description='Angle:', max=180, min=-180, step=5), FloatSlider(value=1…